# Neural Operator Benchmark

Compares our spectral **IntegralOperatorModel** (`model.py`) against three
reference neural operators on the six systems from `equations.py`:

| model | source |
|---|---|
| **Ours** | `model.py` - heat-kernel / Laplace-Beltrami eigenbasis integral operator |
| **CNO** | `ref_models/cno.py` - Convolutional Neural Operator (Raonic et al., NeurIPS 2023) |
| **FNO** | `ref_models/fno.py` - Fourier Neural Operator (Li et al., ICLR 2021) |
| **LNO** | `ref_models/lno.py` - Laplace Neural Operator (Cao et al., Nat. Mach. Intell. 2024) |

**Task: state evolution, no forcing.** Every system is posed the same way -
map an initial state to the evolved state, with no driving term:

- **PDEs** (Diffusion, Reaction-Diffusion, Beam): initial field `u(0)` on the
  spatial grid -> final field `u(T)`.
- **ODEs** (Pendulum, Duffing, Lorenz): initial condition -> the trajectory over
  `n_points` time steps (the IC is broadcast along the time axis so all models
  see the same `[batch, n_points, channels]` interface).

Each model is trained for **1000 epochs** on every system. Outputs: a summary
table of test relative L2 errors, and loss curves per system.

In [1]:
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from scipy.sparse.linalg import eigsh

from equations import (
    GridDomain,
    DiffusionEquation,
    ReactionDiffusion,
    BeamEquation,
    DrivenPendulum,
    DuffingOscillator,
    LorenzSystem,
)
from model import HeatKernel, IntegralOperator, IntegralOperatorModel
from ref_models.cno import CNOOperator
from ref_models.fno import FNOOperator
from ref_models.lno import LNOOperator

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


## Configuration

In [2]:
SEED = 0
N_POINTS = 128          # interior grid points / time steps
N_TOTAL = 128           # samples per system
N_TRAIN = 96
N_EPOCHS = 1000
BATCH_SIZE = 32
LEARNING_RATE = 1e-3

# Our model: Laplacian eigenbasis settings
N_MODES = 32                        # eigenmodes retained
DIFFUSION_TIMES = list(np.linspace(0.005, 1.0, 8))   # heat-kernel time scales
HIDDEN_CHANNELS = 32                # width of the message MLP
LATENT_CHANNELS = 8                 # channels between operator layers
N_OPERATOR_LAYERS = 2               # stacked integral-operator layers

# GridDomain eliminates boundary points, so ask for N_POINTS + 2
domain = GridDomain(lengths=1.0, shape=N_POINTS + 2)
x = domain.points[:, 0]
print("interior points:", domain.n_points)

interior points: 128


## Laplacian eigenpairs (for our model)

Our model works in the eigenbasis of the Laplacian, so we diagonalize the
domain's Laplacian once and reuse it. `domain.laplacian` is negative
semi-definite (the analyst's Laplacian), so we take eigenpairs of `-L`, whose
eigenvalues are the non-negative `lambda_k`.

In [3]:
L = domain.laplacian
eigenvalues, eigenvectors = eigsh(
    (-L).tocsc(), k=N_MODES, sigma=-1e-6, which="LM"
)
order = np.argsort(eigenvalues)
eigenvalues = np.maximum(eigenvalues[order], 0.0)
eigenvectors = eigenvectors[:, order]

# Uniform grid -> uniform mass weights (cells all have the same volume)
mass = np.full(domain.n_points, 1.0 / domain.n_points)

print("first eigenvalues:", np.round(eigenvalues[:5], 3))
print("exact (k*pi)^2:   ", np.round([(k * np.pi) ** 2 for k in range(1, 6)], 3))

first eigenvalues: [  9.869  39.471  88.787 157.789 246.435]
exact (k*pi)^2:    [  9.87   39.478  88.826 157.914 246.74 ]


## Data generation

One dataset per system. PDE inputs are smooth random fields built from the
lowest sine modes; ODE inputs are random initial conditions. Every dataset ends
up shaped `[n_samples, n_points, channels]`.

In [4]:
def make_pde_dataset(equation_factory, n_total=N_TOTAL, T=1.0, seed=SEED):
    """Initial field u(0) -> final field u(T) on the spatial grid."""
    rng = np.random.default_rng(seed)
    modes = np.arange(1, 6)
    u0_list, uT_list = [], []

    for _ in range(n_total):
        coefficients = rng.normal(size=len(modes))
        y0 = sum(
            c * np.sin(m * np.pi * x) for c, m in zip(coefficients, modes)
        )
        y0 = y0 / np.abs(y0).max()

        equation = equation_factory()
        equation.set_initial_condition(y0=y0)
        solution = equation.solve([0.0, T], t_eval=[0.0, T])

        u0_list.append(solution.y[:, 0])
        uT_list.append(solution.y[:, -1])

    u0 = torch.tensor(np.array(u0_list), dtype=torch.float32).unsqueeze(-1)
    uT = torch.tensor(np.array(uT_list), dtype=torch.float32).unsqueeze(-1)
    return u0, uT


def make_beam_dataset(n_total=N_TOTAL, T=0.05, seed=SEED):
    """Beam carries (w, wdot); evolve the displacement field."""
    rng = np.random.default_rng(seed)
    modes = np.arange(1, 4)
    u0_list, uT_list = [], []

    for _ in range(n_total):
        coefficients = rng.normal(size=len(modes))
        w0 = sum(
            c * np.sin(m * np.pi * x) for c, m in zip(coefficients, modes)
        )
        w0 = w0 / np.abs(w0).max()

        equation = BeamEquation(domain, EI=1e-4, rho_A=1.0)
        equation.set_initial_condition(w0=w0, wdot0=np.zeros_like(w0))
        solution = equation.solve([0.0, T], t_eval=[0.0, T])

        n = domain.n_points
        u0_list.append(solution.y[:n, 0])
        uT_list.append(solution.y[:n, -1])

    u0 = torch.tensor(np.array(u0_list), dtype=torch.float32).unsqueeze(-1)
    uT = torch.tensor(np.array(uT_list), dtype=torch.float32).unsqueeze(-1)
    return u0, uT


def make_ode_dataset(equation_factory, state_ranges, T, n_total=N_TOTAL,
                     seed=SEED, rtol=1e-9, atol=1e-11):
    """Initial condition -> trajectory over N_POINTS time steps.

    The initial state is broadcast along the time axis so the input has the
    same [n_samples, n_points, channels] layout as the PDE tasks.
    """
    rng = np.random.default_rng(seed)
    t_eval = np.linspace(0.0, T, N_POINTS)
    inputs, outputs = [], []

    for _ in range(n_total):
        state0 = [rng.uniform(lo, hi) for lo, hi in state_ranges]

        equation = equation_factory()
        equation.set_initial_condition(*state0)
        solution = equation.solve(
            [0.0, T], t_eval=t_eval, rtol=rtol, atol=atol
        )

        inputs.append(np.tile(np.array(state0)[None, :], (N_POINTS, 1)))
        outputs.append(solution.y.T)

    X = torch.tensor(np.array(inputs), dtype=torch.float32)
    Y = torch.tensor(np.array(outputs), dtype=torch.float32)
    return X, Y

In [5]:
datasets = {}

datasets["Diffusion"] = make_pde_dataset(
    lambda: DiffusionEquation(domain, D=0.02), T=1.0
)

datasets["Reaction-Diffusion"] = make_pde_dataset(
    lambda: ReactionDiffusion(domain, D=0.02, k=0.5), T=1.0
)

datasets["Beam"] = make_beam_dataset(T=0.05)

datasets["Pendulum"] = make_ode_dataset(
    lambda: DrivenPendulum(c=0.5),
    state_ranges=[(-1.5, 1.5), (-1.5, 1.5)],
    T=10.0,
)

datasets["Duffing"] = make_ode_dataset(
    lambda: DuffingOscillator(c=0.5),
    state_ranges=[(-1.5, 1.5), (-1.5, 1.5)],
    T=10.0,
)

datasets["Lorenz"] = make_ode_dataset(
    lambda: LorenzSystem(rho=28.0),
    state_ranges=[(-1.0, 1.0), (-1.0, 1.0), (20.0, 30.0)],
    T=2.0,
)

for name, (X, Y) in datasets.items():
    print(f"{name:20s} X {tuple(X.shape)}  Y {tuple(Y.shape)}")

Diffusion            X (128, 128, 1)  Y (128, 128, 1)
Reaction-Diffusion   X (128, 128, 1)  Y (128, 128, 1)
Beam                 X (128, 128, 1)  Y (128, 128, 1)
Pendulum             X (128, 128, 2)  Y (128, 128, 2)
Duffing              X (128, 128, 2)  Y (128, 128, 2)
Lorenz               X (128, 128, 3)  Y (128, 128, 3)


### Normalization

Lorenz in particular lives on a very different scale from the PDE fields, so we
standardize each dataset using **training-set statistics only** (no test
leakage). Errors are reported in the normalized space, which makes them
comparable across systems.

In [6]:
def normalize(X, Y, n_train=N_TRAIN):
    x_mean, x_std = X[:n_train].mean(), X[:n_train].std()
    y_mean, y_std = Y[:n_train].mean(), Y[:n_train].std()
    x_std = torch.clamp(x_std, min=1e-8)
    y_std = torch.clamp(y_std, min=1e-8)
    return (X - x_mean) / x_std, (Y - y_mean) / y_std


datasets = {
    name: normalize(X, Y) for name, (X, Y) in datasets.items()
}
print("normalized using training-set statistics")

normalized using training-set statistics


## Model builders

All four take `[batch, n_points, channels]` and return the same shape, so they
are interchangeable. Our model needs the Laplacian eigenpairs; CNO/FNO/LNO are
constructed with their reference defaults.

In [7]:
def build_ours(in_channels, out_channels):
    layers = []

    for layer_index in range(N_OPERATOR_LAYERS):
        layer_in = in_channels if layer_index == 0 else LATENT_CHANNELS
        layer_out = (
            out_channels
            if layer_index == N_OPERATOR_LAYERS - 1
            else LATENT_CHANNELS
        )

        kernel = HeatKernel(
            eigenvalues=torch.tensor(eigenvalues, dtype=torch.float32),
            eigenvectors=torch.tensor(eigenvectors, dtype=torch.float32),
            mass=mass,
            in_channels=layer_in,
            out_channels=layer_out,
            diffusion_times=DIFFUSION_TIMES,
        )
        layers.append(
            IntegralOperator(kernel, hidden_channels=HIDDEN_CHANNELS)
        )

    return IntegralOperatorModel(layers)


def build_cno(in_channels, out_channels):
    return CNOOperator(
        n_points=N_POINTS,
        in_channels=in_channels,
        out_channels=out_channels,
        N_layers=3,
        N_res=2,
        N_res_neck=2,
        channel_multiplier=16,
    )


def build_fno(in_channels, out_channels):
    return FNOOperator(
        n_points=N_POINTS,
        in_channels=in_channels,
        out_channels=out_channels,
        modes=16,
        width=64,
    )


def build_lno(in_channels, out_channels):
    return LNOOperator(
        n_points=N_POINTS,
        in_channels=in_channels,
        out_channels=out_channels,
        modes=16,
        width=4,
        coords=x,
    )


MODEL_BUILDERS = {
    "Ours": build_ours,
    "CNO": build_cno,
    "FNO": build_fno,
    "LNO": build_lno,
}

## Training loop

Identical protocol for every model and system: Adam, mini-batches, 1000 epochs,
with the learning rate halved every 250 epochs. We record the training loss and
the test relative L2 error each epoch.

In [8]:
def relative_l2(prediction, target):
    return (
        torch.norm(prediction - target) / torch.norm(target)
    ).item()


def train_model(model, X, Y, n_epochs=N_EPOCHS, seed=SEED):
    torch.manual_seed(seed)
    model = model.to(device)

    X_train, Y_train = X[:N_TRAIN].to(device), Y[:N_TRAIN].to(device)
    X_test, Y_test = X[N_TRAIN:].to(device), Y[N_TRAIN:].to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=250, gamma=0.5
    )

    history = {"train_loss": [], "test_error": []}
    n_batches = max(1, N_TRAIN // BATCH_SIZE)

    for epoch in range(n_epochs):
        model.train()
        permutation = torch.randperm(N_TRAIN, device=device)
        epoch_loss = 0.0

        for b in range(n_batches):
            index = permutation[b * BATCH_SIZE:(b + 1) * BATCH_SIZE]
            optimizer.zero_grad()
            loss = nn.functional.mse_loss(model(X_train[index]), Y_train[index])
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        scheduler.step()

        model.eval()
        with torch.no_grad():
            test_error = relative_l2(model(X_test), Y_test)

        history["train_loss"].append(epoch_loss / n_batches)
        history["test_error"].append(test_error)

    return history

## Run the benchmark

This trains 4 models x 6 systems x 1000 epochs. Expect this cell to take a
while on CPU.

In [9]:
results = {}
histories = {}
trained_models = {}

for system_name, (X, Y) in datasets.items():
    in_channels = X.shape[-1]
    out_channels = Y.shape[-1]
    results[system_name] = {}
    histories[system_name] = {}
    trained_models[system_name] = {}

    print(f"\n=== {system_name} "
          f"(in={in_channels}, out={out_channels}) ===")

    for model_name, builder in MODEL_BUILDERS.items():
        torch.manual_seed(SEED)
        model = builder(in_channels, out_channels)
        n_parameters = sum(p.numel() for p in model.parameters())

        start = time.time()
        history = train_model(model, X, Y)
        elapsed = time.time() - start

        final_error = history["test_error"][-1]
        best_error = min(history["test_error"])

        results[system_name][model_name] = {
            "final_error": final_error,
            "best_error": best_error,
            "n_parameters": n_parameters,
            "seconds": elapsed,
        }
        histories[system_name][model_name] = history
        trained_models[system_name][model_name] = model

        print(f"  {model_name:5s} params={n_parameters:8d}  "
              f"final rel L2 {final_error:.4f}  "
              f"best {best_error:.4f}  ({elapsed:.0f}s)")


=== Diffusion (in=1, out=1) ===


  Ours  params=    1065  final rel L2 0.1983  best 0.1983  (61s)


  CNO   params=  107081  final rel L2 0.0468  best 0.0407  (77s)


  FNO   params=  287425  final rel L2 0.0050  best 0.0050  (33s)


  LNO   params=    1313  final rel L2 0.4436  best 0.4294  (26s)

=== Reaction-Diffusion (in=1, out=1) ===


  Ours  params=    1065  final rel L2 0.2067  best 0.2067  (49s)


... (15 outputs left out)

  Ours  params=    1515  final rel L2 0.6677  best 0.6677  (60s)


  CNO   params=  107851  final rel L2 0.2395  best 0.1998  (80s)


  FNO   params=  287811  final rel L2 0.1526  best 0.1500  (29s)


  LNO   params=    1579  final rel L2 0.4025  best 0.3969  (22s)


## Results table

Test relative L2 error after 1000 epochs (lower is better); the best model for
each system is marked with `*`.

In [10]:
model_names = list(MODEL_BUILDERS.keys())

header = f"{'System':<22}" + "".join(f"{m:>14}" for m in model_names)
print(header)
print("-" * len(header))

for system_name in datasets:
    row = f"{system_name:<22}"
    errors = {
        m: results[system_name][m]["final_error"] for m in model_names
    }
    best_model = min(errors, key=errors.get)

    for model_name in model_names:
        error = errors[model_name]
        mark = "*" if model_name == best_model else " "
        row += f"{error:>13.4f}{mark}"
    print(row)

print("\nParameter counts")
print(f"{'System':<22}" + "".join(f"{m:>14}" for m in model_names))
for system_name in datasets:
    row = f"{system_name:<22}"
    for model_name in model_names:
        row += f"{results[system_name][model_name]['n_parameters']:>14d}"
    print(row)

System                          Ours           CNO           FNO           LNO
------------------------------------------------------------------------------
Diffusion                    0.1983        0.0468        0.0050*       0.4436 
Reaction-Diffusion           0.2067        0.0577        0.0074*       0.4657 
Beam                         0.3286        0.0087        0.0016*       0.1933 
Pendulum                     0.9350        0.0514        0.0089*       0.0866 
Duffing                      0.8827        0.0492        0.0067*       0.3378 
Lorenz                       0.6677        0.2395        0.1526*       0.4025 

Parameter counts
System                          Ours           CNO           FNO           LNO
Diffusion                       1065        107081        287425          1313
Reaction-Diffusion              1065        107081        287425          1313
Beam                            1065        107081        287425          1313
Pendulum                        12

### Same table as a DataFrame

Optional - only runs if pandas is installed. Best model per row is highlighted.

In [11]:
try:
    import pandas as pd

    table = pd.DataFrame(
        {
            model_name: {
                system_name: results[system_name][model_name]["final_error"]
                for system_name in datasets
            }
            for model_name in model_names
        }
    )
    table.index.name = "System"
    display(
        table.style.format("{:.4f}").highlight_min(axis=1, color="#c6efce")
    )
except ImportError:
    print("pandas not installed - see the printed table above instead.")

,Ours,CNO,FNO,LNO
System,,,,
Diffusion,0.1983,0.0468,0.0050,0.4436
Reaction-Diffusion,0.2067,0.0577,0.0074,0.4657
Beam,0.3286,0.0087,0.0016,0.1933
Pendulum,0.9350,0.0514,0.0089,0.0866
Duffing,0.8827,0.0492,0.0067,0.3378
Lorenz,0.6677,0.2395,0.1526,0.4025


## Loss curves

Training loss and test error per system, one panel per system.

In [12]:
colors = {"Ours": "#d62728", "CNO": "#1f77b4", "FNO": "#2ca02c",
          "LNO": "#ff7f0e"}

n_systems = len(datasets)
fig, axes = plt.subplots(2, n_systems, figsize=(4.2 * n_systems, 7.5))

for column, system_name in enumerate(datasets):
    ax_loss = axes[0, column]
    ax_error = axes[1, column]

    for model_name in model_names:
        history = histories[system_name][model_name]
        ax_loss.semilogy(
            history["train_loss"], label=model_name,
            color=colors[model_name], linewidth=1.2,
        )
        ax_error.semilogy(
            history["test_error"], label=model_name,
            color=colors[model_name], linewidth=1.2,
        )

    ax_loss.set_title(system_name)
    ax_loss.set_xlabel("Epoch")
    ax_error.set_xlabel("Epoch")
    if column == 0:
        ax_loss.set_ylabel("Training MSE")
        ax_error.set_ylabel("Test relative L2")
    ax_loss.legend(fontsize=8)
    ax_loss.grid(alpha=0.3)
    ax_error.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Example predictions

Ground truth against each trained model's prediction on one held-out test
sample (first output channel).

In [13]:
fig, axes = plt.subplots(2, (len(datasets) + 1) // 2, figsize=(16, 7.5))
axes = axes.ravel()

sample = 0

for panel, system_name in enumerate(datasets):
    X, Y = datasets[system_name]
    ax = axes[panel]

    target = Y[N_TRAIN + sample, :, 0].numpy()
    ax.plot(target, "k-", linewidth=2.4, label="Ground truth")

    for model_name in model_names:
        model = trained_models[system_name][model_name]
        model.eval()
        with torch.no_grad():
            prediction = model(
                X[N_TRAIN:N_TRAIN + 1].to(device)
            )[0, :, 0].cpu().numpy()
        ax.plot(
            prediction, "--", linewidth=1.4,
            color=colors[model_name], label=model_name,
        )

    ax.set_title(system_name)
    ax.set_xlabel("Grid point / time step")
    ax.grid(alpha=0.3)
    if panel == 0:
        ax.set_ylabel("u")
        ax.legend(fontsize=8)

for panel in range(len(datasets), len(axes)):
    axes[panel].axis("off")

plt.tight_layout()
plt.show()